# Backtesting Showcase — AI Trading Bot Pro Max Ultra Plus 9000

This notebook demonstrates a **quant-style evaluation** layer built on top of the existing project.

The key idea: **a great prediction is not a strategy**.

A strategy requires:

- rules for converting signals into positions
- execution assumptions (e.g. next-bar)
- trading frictions (fees/slippage)
- portfolio accounting (equity + drawdown)
- evaluation metrics (Sharpe/Sortino/Calmar)

We reuse the project’s **existing unified score** (`final_score`) as the signal source of truth.


## 1) Setup

Run this from the repo root after installing the package:

```bash
pip install -e '.[api]'
```


In [ ]:
from btc_paper.config import load_settings
from btc_paper.backtest.dataset import generate_backtest_dataset
from btc_paper.backtest.engine import run_backtest
from btc_paper.backtest.schemas import BacktestParams

settings = load_settings()
ds = generate_backtest_dataset(settings)
ds.df.head(), len(ds.df)


## 2) Baseline backtest: fixed sizing, no costs

Assumption: the signal at bar *t* is **executed at bar *t+1***.

This avoids lookahead bias (you cannot trade on a candle close **before** it exists).


In [ ]:
params = BacktestParams(
    sizing_mode="fixed",
    buy_threshold=0.35,
    sell_threshold=-0.35,
    fee_bps=0.0,
    slippage_bps=0.0,
)
res = run_backtest(ds.df, params)
res.summary


## 3) Add trading friction (fees + slippage)

Costs are applied only when the executed position changes, using a **turnover-based** model.


In [ ]:
res_cost = run_backtest(ds.df, BacktestParams(fee_bps=5.0, slippage_bps=5.0))
res_cost.summary


## 4) Position sizing variants

We support three sizing modes:

- **fixed**: exposure = ±1 or 0
- **confidence**: exposure scales with `abs(final_score)`
- **confidence_vol**: confidence scaled by target-vol / realized-vol (rolling std of returns)


In [ ]:
modes = ["fixed", "confidence", "confidence_vol"]
rows = []
for m in modes:
    r = run_backtest(ds.df, BacktestParams(sizing_mode=m, fee_bps=5.0, slippage_bps=5.0))
    rows.append((m, r.summary.cumulative_return, r.summary.sharpe, r.summary.max_drawdown, r.summary.total_cost_paid))
rows


## 5) Walk-forward (out-of-sample) evaluation

A stricter evaluation is to test in rolling windows.

This first pass does **not** retrain models yet — it evaluates the historical unified score in sequential test windows and concatenates the out-of-sample equity curve.


In [ ]:
from btc_paper.backtest.schemas import BacktestParams
from btc_paper.backtest.engine import run_backtest

train_bars = 24 * 30
test_bars = 24 * 7
step_bars = 24 * 7

df = ds.df
eq0 = 10_000.0
oos = []
windows = []
i = 0
while True:
    train_end = i + train_bars
    test_end = train_end + test_bars
    if test_end > len(df):
        break
    test_df = df.iloc[train_end:test_end]
    r = run_backtest(test_df, BacktestParams(initial_capital=eq0, fee_bps=5.0, slippage_bps=5.0))
    if r.equity_curve:
        eq0 = float(r.equity_curve[-1]["equity"])
    windows.append(r.summary)
    oos.extend(r.equity_curve)
    i += step_bars

len(windows), (float(oos[-1]["equity"]) / 10_000.0 - 1.0) if oos else None


## Final notes / limitations

- If your SQLite history has few stored signals (`final_score`), the backtest will look flat.
- For a portfolio-grade history, let the scheduler run for days/weeks, or add a historical reconstruction job.
- Walk-forward model retraining can be added next (Phase 4+) and plugged into the same engine.
